# Use SRGAN

This source code is used to **generate Super-Resolution images from the given Low-Resolution (LR) images** using the **pre-trained SRGAN model**.

---

# 1. Import library

In [ ]:
import os
import sys
import cv2
import torch
import random
import importlib
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
from torchvision.transforms import ToTensor, ToPILImage

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# 2. Setup

### 2.1. PATH

In [ ]:
INPUT_PATH = "dataset"
OUTPUT_PATH = "dataset_SRGAN"

MODEL_PATH = "result/model.pth"
GENERATOR_PATH = "utils"

def ensure_dir(path):
    if not os.path.exists(path):
        os.makedirs(path)
        print(f"Created folder: {path}")
    else:
        print(f"Folder exists: {path}")

ensure_dir(INPUT_PATH)
ensure_dir(OUTPUT_PATH)

### 2.2. Config

Adjust the **HR_SIZE**

In [ ]:
HR_SIZE = 224
SCALE = 4

print("high resolution :", HR_SIZE)
print("scale :", SCALE)

### 2.3. Load model

In [ ]:
def load_generator(model_path, gen):

    sys.path.append(gen)

    module = importlib.import_module("model")
    importlib.reload(module)

    Generator = module.Generator

    print(f"Loaded model file: {gen}/model.py")

    generator = Generator(SCALE).to(device)

    weight_path = f"{model_path}"
    state_dict = torch.load(weight_path, map_location=device)

    generator.load_state_dict(state_dict, strict=False)
    generator.eval()

    print("Model loaded successfully.")

    return generator

# 3. Utils

### 3.1. Process

In [ ]:
def process_folder(input_dir, output_dir, generator):
    os.makedirs(output_dir, exist_ok=True)

    image_extensions = (".png", ".jpg", ".jpeg", ".bmp")
    files = [f for f in os.listdir(input_dir) if f.lower().endswith(image_extensions)]

    print(f"Found {len(files)} images")

    for filename in tqdm(files, desc="Processing Images"):
        input_path = os.path.join(input_dir, filename)
        output_path = os.path.join(output_dir, filename)

        img = cv2.imread(input_path)
        if img is None:
            continue

        # Original image (no LR resize)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # Convert to tensor
        lr_tensor = ToTensor()(img_rgb).unsqueeze(0).to(device)

        # Run SRGAN
        with torch.no_grad():
            sr_tensor = generator(lr_tensor)

        # Post-process
        sr_tensor = sr_tensor.squeeze(0).cpu().clamp(0, 1)
        sr_img = ToPILImage()(sr_tensor)
        sr_img = np.array(sr_img)

        # Resize AFTER SRGAN
        sr_final = cv2.resize(
            sr_img,
            (HR_SIZE, HR_SIZE),
            interpolation=cv2.INTER_CUBIC
        )

        # Save output
        sr_bgr = cv2.cvtColor(sr_final, cv2.COLOR_RGB2BGR)
        cv2.imwrite(output_path, sr_bgr)

    print("Finished")

### 3.2. Save

In [ ]:
def save_random_grid(input_dir, output_dir, save_path="comparison_grid.png", num_samples=4):
    
    image_extensions = (".png", ".jpg", ".jpeg", ".bmp")
    files = [f for f in os.listdir(input_dir) if f.lower().endswith(image_extensions)]

    if len(files) == 0:
        print("No images found")
        return

    num_samples = min(num_samples, len(files))
    selected_files = random.sample(files, num_samples)

    # make figure
    fig = plt.figure(figsize=(10, 5 * num_samples))

    # column head
    fig.text(0.25, 0.96, "Low Resolution (Input)", 
             ha='center', va='center', fontsize=16, fontweight='bold')
    fig.text(0.75, 0.96, "Super Resolution (SRGAN)", 
             ha='center', va='center', fontsize=16, fontweight='bold')

    for i, filename in enumerate(selected_files):

        lr = cv2.imread(os.path.join(input_dir, filename))
        sr = cv2.imread(os.path.join(output_dir, filename))

        lr = cv2.cvtColor(lr, cv2.COLOR_BGR2RGB)
        sr = cv2.cvtColor(sr, cv2.COLOR_BGR2RGB)

        # LR
        plt.subplot(num_samples, 2, 2*i + 1)
        plt.imshow(lr)
        plt.title(f"{filename}\n{lr.shape}")
        plt.axis("off")

        # SR
        plt.subplot(num_samples, 2, 2*i + 2)
        plt.imshow(sr)
        plt.title(f"{sr.shape}")
        plt.axis("off")

    plt.tight_layout(rect=[0, 0, 1, 0.94])

    plt.savefig(save_path, dpi=300)
    print(f"Grid saved to: {save_path}")

    plt.show()

#### 3.3. Using SRGAN function

In [ ]:
def process_all_classes(input_dir, output_dir, model_dir, generator):

        generator = load_generator(
            model_path=model_dir,
            gen = generator
        )

        os.makedirs(output_dir, exist_ok=True)
        process_folder(input_dir, output_dir, generator)
        
        save_random_grid(
            input_dir=input_dir,
            output_dir=output_dir,
            save_path="example.png",
            num_samples=4
        )

# 4. Using SRGAN

In [ ]:
process_all_classes(
        INPUT_PATH,
        OUTPUT_PATH,
        MODEL_PATH,
        GENERATOR_PATH
    )